# Compare DistilBERT Model Variants

This notebook compares the test performance of three saved DistilBERT models:
- baseline full conversation
- customer-only text
- customer-only text with less max token length
- metadata-prefix text
- customer-only text with less learning rate

The goal is to compare their metrics side by side and understand what each evaluation metric means.

In [34]:
import numpy as np
import pandas as pd
import torch

from torch.utils.data import Dataset, DataLoader
from transformers import DistilBertForSequenceClassification
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix

In [35]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("Using device:", device)

Using device: mps


## Models

In [36]:
model_variants = {
    "baseline": {
        "model_path": "best_distilbert_model_baseline.pt",
        "input_ids_path": "X_test_input_ids.npy",
        "attention_mask_path": "X_test_attention_mask.npy",
        "labels_path": "y_test.npy"
    },
    "customer_only": {
        "model_path": "best_distilbert_model_customer_only.pt",
        "input_ids_path": "X_test_input_ids_customer_only.npy",
        "attention_mask_path": "X_test_attention_mask_customer_only.npy",
        "labels_path": "y_test_customer_only.npy"
    },
    "customer_only_386": {
        "model_path": "best_distilbert_model_customer_only_386.pt",
        "input_ids_path": "X_test_input_ids_customer_only_386.npy",
        "attention_mask_path": "X_test_attention_mask_customer_only_386.npy",
        "labels_path": "y_test_customer_only_386.npy"
    },
    "metadata_prefix": {
        "model_path": "best_distilbert_model_metadata_prefix.pt",
        "input_ids_path": "X_test_input_ids_metadata_prefix.npy",
        "attention_mask_path": "X_test_attention_mask_metadata_prefix.npy",
        "labels_path": "y_test_metadata_prefix.npy"
    },
    "customer_only_low_lr": {
        "model_path": "best_distilbert_model_customer_only_low_lr.pt",
        "input_ids_path": "X_test_input_ids_customer_only.npy",
        "attention_mask_path": "X_test_attention_mask_customer_only.npy",
        "labels_path": "y_test_customer_only.npy"
    }
}

label_names = ["negative", "neutral", "positive"]
model_checkpoint = "distilbert-base-uncased"
num_labels = 3

## Class to read input_ids, attention_mask, labels together

In [28]:
class SentimentDataset(Dataset):
    def __init__(self, input_ids, attention_mask, labels):
        self.input_ids = torch.tensor(input_ids, dtype=torch.long)
        self.attention_mask = torch.tensor(attention_mask, dtype=torch.long)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids": self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
            "labels": self.labels[idx]
        }

In [29]:
def evaluate_model(model, data_loader, device):
    model.eval()
    all_preds = []
    all_labels = []
    total_loss = 0
    loss_fn = torch.nn.CrossEntropyLoss()

    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            loss = loss_fn(logits, labels)

            preds = torch.argmax(logits, dim=1)
            total_loss += loss.item()

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(data_loader)
    accuracy = accuracy_score(all_labels, all_preds)
    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        all_labels,
        all_preds,
        average="macro",
        zero_division=0
    )
    precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
        all_labels,
        all_preds,
        average="weighted",
        zero_division=0
    )

    return {
        "loss": avg_loss,
        "accuracy": accuracy,
        "precision_macro": precision_macro,
        "recall_macro": recall_macro,
        "f1_macro": f1_macro,
        "precision_weighted": precision_weighted,
        "recall_weighted": recall_weighted,
        "f1_weighted": f1_weighted,
        "y_true": all_labels,
        "y_pred": all_preds
    }

## Evaluate all saved models

In [30]:
all_results = {}

for model_name, paths in model_variants.items():
    print(f"Evaluating: {model_name}")

    X_test_input_ids = np.load(paths["input_ids_path"])
    X_test_attention_mask = np.load(paths["attention_mask_path"])
    y_test = np.load(paths["labels_path"])

    test_dataset = SentimentDataset(X_test_input_ids, X_test_attention_mask, y_test)
    test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

    model = DistilBertForSequenceClassification.from_pretrained(model_checkpoint, num_labels=num_labels)
    model.load_state_dict(torch.load(paths["model_path"], map_location=device))
    model.to(device)

    all_results[model_name] = evaluate_model(model, test_loader, device)

Evaluating: baseline


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 6306.28it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Evaluating: customer_only


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 15337.35it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Evaluating: customer_only_386


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 9438.34it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Evaluating: metadata_prefix


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 13101.06it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Evaluating: customer_only_low_lr


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 8380.23it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## Compare metrics

In [31]:
comparison_df = pd.DataFrame({
    model_name: {
        "test_accuracy": result["accuracy"],
        "test_f1_macro": result["f1_macro"],
        "test_precision_macro": result["precision_macro"],
        "test_recall_macro": result["recall_macro"],
        "test_f1_weighted": result["f1_weighted"],
        "test_loss": result["loss"]
    }
    for model_name, result in all_results.items()
}).T

display(comparison_df.sort_values("test_f1_macro", ascending=False))

,test_accuracy,test_f1_macro,test_precision_macro,test_recall_macro,test_f1_weighted,test_loss
customer_only_386,0.566667,0.539954,0.761905,0.566667,0.539954,1.601847
customer_only,0.566667,0.513414,0.767857,0.566667,0.513414,1.612123
baseline,0.600000,0.500000,0.450000,0.600000,0.500000,1.402400
metadata_prefix,0.566667,0.469622,0.413078,0.566667,0.469622,1.805416
customer_only_low_lr,0.500000,0.417112,0.416149,0.500000,0.417112,2.091037


This table gives the main comparison. For this task, **macro F1** is usually the most important metric because the sentiment classes are imbalanced.

- **accuracy -->** overall proportion of correct predictions
- **precision_macro -->** average class-wise precision, treating each class equally
- **recall_macro -->** average class-wise recall, treating each class equally
- **f1_macro -->** average class-wise F1-score, treating each class equally
- **f1_weighted-->** F1-score weighted by class frequency
- **loss -->** model error value; lower is better, but classification metrics are usually easier to interpret

In [32]:
for model_name, result in all_results.items():
    print(f"\n===== {model_name.upper()} =====")
    print(classification_report(result["y_true"], result["y_pred"], target_names=label_names, zero_division=0))


===== BASELINE =====
              precision    recall  f1-score   support

    negative       0.90      0.90      0.90        10
     neutral       0.45      0.90      0.60        10
    positive       0.00      0.00      0.00        10

    accuracy                           0.60        30
   macro avg       0.45      0.60      0.50        30
weighted avg       0.45      0.60      0.50        30


===== CUSTOMER_ONLY =====
              precision    recall  f1-score   support

    negative       0.88      0.70      0.78        10
     neutral       0.43      0.90      0.58        10
    positive       1.00      0.10      0.18        10

    accuracy                           0.57        30
   macro avg       0.77      0.57      0.51        30
weighted avg       0.77      0.57      0.51        30


===== CUSTOMER_ONLY_386 =====
              precision    recall  f1-score   support

    negative       0.86      0.60      0.71        10
     neutral       0.43      0.90      0.58      

## Confusion matrices

In [33]:
for model_name, result in all_results.items():
    cm = confusion_matrix(result["y_true"], result["y_pred"])
    cm_df = pd.DataFrame(cm, index=label_names, columns=label_names)
    print(f"\n===== CONFUSION MATRIX: {model_name.upper()} =====")
    display(cm_df)


===== CONFUSION MATRIX: BASELINE =====


,negative,neutral,positive
negative,9,1,0
neutral,1,9,0
positive,0,10,0



===== CONFUSION MATRIX: CUSTOMER_ONLY =====


,negative,neutral,positive
negative,7,3,0
neutral,1,9,0
positive,0,9,1



===== CONFUSION MATRIX: CUSTOMER_ONLY_386 =====


,negative,neutral,positive
negative,6,4,0
neutral,1,9,0
positive,0,8,2



===== CONFUSION MATRIX: METADATA_PREFIX =====


,negative,neutral,positive
negative,9,1,0
neutral,2,8,0
positive,0,10,0



===== CONFUSION MATRIX: CUSTOMER_ONLY_LOW_LR =====


,negative,neutral,positive
negative,6,4,0
neutral,1,9,0
positive,0,10,0


## Final note

Use `test_f1_macro` as the main comparison metric, because it gives equal importance to all three sentiment classes. If two models have the same test accuracy, the one with the higher macro F1 is usually the better balanced model.